# Module 7 : Interfaces interactives avec ipywidgets

## Objectifs
- Comprendre ce qu'est une interface interactive
- Créer des widgets dans un notebook (curseurs, boutons, listes…)
- Utiliser `@interact` pour rendre une fonction interactive en 1 ligne
- Construire un mini-outil marketing interactif

## 7.1 Pourquoi des interfaces interactives ?

Jusqu'ici, pour tester une fonction avec d'autres valeurs, vous deviez **modifier le code** et ré-exécuter la cellule. Avec **ipywidgets**, vous ajoutez des **curseurs**, **champs de texte** ou **listes déroulantes** qui modifient les paramètres en temps réel.

**Cas d'usage marketing :**
- Simulateur de ROI où le directeur marketing fait varier le budget
- Calculateur de prix TTC avec choix de la TVA
- Scoring de leads interactif pour une démo client

## 7.2 Installation

ipywidgets est généralement déjà installé avec Anaconda / Jupyter. Si l'import échoue avec `ModuleNotFoundError`, **exécutez la cellule ci-dessous** : la commande magique `%pip` installe le module dans le **même Python que ce notebook** (peu importe l'éditeur : Jupyter, VS Code…).

⚠️ **Après l'installation, redémarrez le kernel** (VS Code : icône ↻ en haut du notebook ; Jupyter : menu *Kernel → Restart*) puis ré-exécutez la cellule d'import.

In [3]:
# À exécuter UNE seule fois si ipywidgets n'est pas installé
# Puis redémarrez le kernel avant de continuer
%pip install ipywidgets


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
# Import de la librairie
import ipywidgets as widgets
from IPython.display import display

print("ipywidgets prêt !")

ipywidgets prêt !


## 7.3 Votre premier widget : le curseur (slider)

Un slider permet à l'utilisateur de choisir une valeur entre un minimum et un maximum.

In [5]:
# Créer un slider qui va de 0 à 100
slider_budget = widgets.IntSlider(
    value=5000,
    min=0,
    max=20000,
    step=500,
    description="Budget (€) :"
)

display(slider_budget)

IntSlider(value=5000, description='Budget (€) :', max=20000, step=500)

In [ ]:
# On peut récupérer la valeur sélectionnée
print(f"Budget choisi : {slider_budget.value} €")

## 7.4 La fonction magique : `@interact`

C'est la manière la plus simple de rendre une fonction interactive : un **décorateur** placé au-dessus d'une fonction crée automatiquement les widgets correspondants.

In [ ]:
from ipywidgets import interact

@interact(prix_ht=(0, 1000, 10), tva=(0, 25, 0.5))
def calculer_ttc(prix_ht=100, tva=20):
    prix_ttc = prix_ht * (1 + tva / 100)
    print(f"Prix HT : {prix_ht} €")
    print(f"TVA : {tva} %")
    print(f"Prix TTC : {prix_ttc:.2f} €")

Bougez les curseurs : la fonction est ré-exécutée automatiquement à chaque changement.

**Comment lire `(0, 1000, 10)` ?** → (min, max, pas).

## 7.5 Les principaux widgets

| Widget | Utilisation |
|--------|-------------|
| `IntSlider` / `FloatSlider` | Curseur numérique |
| `Text` | Champ de texte court |
| `Textarea` | Texte long |
| `Dropdown` | Liste déroulante |
| `Checkbox` | Case à cocher |
| `Button` | Bouton cliquable |
| `DatePicker` | Sélecteur de date |

In [ ]:
# Un dropdown (liste déroulante)
@interact(canal=["email", "sms", "display", "social"])
def afficher_canal(canal):
    print(f"Canal sélectionné : {canal}")

In [ ]:
# Combinaison de plusieurs types de widgets
@interact(
    nom="Black Friday",
    budget=(500, 10000, 100),
    canal=["email", "sms", "display"],
    active=True
)
def afficher_campagne(nom, budget, canal, active):
    statut = "✅ active" if active else "⏸ inactive"
    print(f"Campagne : {nom}")
    print(f"Budget   : {budget} €")
    print(f"Canal    : {canal}")
    print(f"Statut   : {statut}")

## 7.6 Exemple Marketing : Simulateur de ROI

On reprend la fonction `calculer_roi` du module 6 et on la rend interactive.

In [ ]:
@interact(
    budget=(100, 20000, 100),
    revenus=(0, 50000, 500)
)
def simuler_roi(budget=5000, revenus=15000):
    if budget == 0:
        print("Le budget ne peut pas être 0")
        return
    roi = ((revenus - budget) / budget) * 100
    print(f"Budget   : {budget} €")
    print(f"Revenus  : {revenus} €")
    print(f"ROI      : {roi:.1f} %")
    if roi > 100:
        print("🚀 Excellente campagne !")
    elif roi > 0:
        print("👍 Campagne rentable")
    else:
        print("⚠️ Campagne déficitaire")

## 7.7 Boutons et événements

Pour des actions ponctuelles (ex : valider un formulaire), on utilise un **bouton** lié à une fonction.

In [ ]:
bouton = widgets.Button(description="Calculer le ROI", button_style="primary")
sortie = widgets.Output()

champ_budget = widgets.FloatText(value=5000, description="Budget :")
champ_revenus = widgets.FloatText(value=15000, description="Revenus :")

def au_clic(b):
    with sortie:
        sortie.clear_output()
        roi = ((champ_revenus.value - champ_budget.value) / champ_budget.value) * 100
        print(f"ROI = {roi:.1f} %")

bouton.on_click(au_clic)

display(champ_budget, champ_revenus, bouton, sortie)

## 7.8 Organiser les widgets : VBox et HBox

- **VBox** : empile les widgets verticalement
- **HBox** : aligne les widgets horizontalement

In [ ]:
prix = widgets.FloatSlider(value=100, min=0, max=500, description="Prix HT :")
tva = widgets.Dropdown(options=[5.5, 10, 20], value=20, description="TVA :")
resultat = widgets.Label(value="")

def maj(*args):
    ttc = prix.value * (1 + tva.value / 100)
    resultat.value = f"Prix TTC : {ttc:.2f} €"

prix.observe(maj, names="value")
tva.observe(maj, names="value")
maj()

interface = widgets.VBox([
    widgets.HBox([prix, tva]),
    resultat
])
display(interface)

## 7.9 Exemple complet : Scoring de lead interactif

On reprend la logique du module 6 et on en fait un mini-outil utilisable par un commercial.

In [ ]:
@interact(
    email=["perso (gmail/yahoo…)", "professionnel"],
    pages_vues=(0, 30, 1),
    minutes_passees=(0, 60, 1),
    a_telecharge=False,
    taille_entreprise=(1, 1000, 10),
    secteur=["Tech", "Finance", "Retail", "Autre"]
)
def scorer(email="professionnel", pages_vues=5, minutes_passees=5,
           a_telecharge=False, taille_entreprise=100, secteur="Tech"):
    score = 0
    score += 5 if email.startswith("perso") else 20
    score += min(pages_vues * 2, 20)
    score += min(minutes_passees, 15)
    if a_telecharge:
        score += 25
    if taille_entreprise >= 500:
        score += 30
    elif taille_entreprise >= 100:
        score += 20
    elif taille_entreprise >= 20:
        score += 10
    if secteur in ["Tech", "Finance", "Retail"]:
        score += 15

    if score >= 80:
        qualif = "🔥 HOT — contacter sous 24h"
    elif score >= 50:
        qualif = "🌤 WARM — nurturing prioritaire"
    else:
        qualif = "❄️ COLD — à surveiller"

    print(f"Score : {score} / 110")
    print(qualif)

## Exercices pratiques

### Exercice 1
Créez un widget interactif qui affiche le prix après une remise. Paramètres : `prix` (slider 0–500) et `pourcentage` (slider 0–80).

In [ ]:
# Votre code ici


### Exercice 2
Créez un simulateur de **coût d'acquisition client (CAC)** :
- `budget` : slider 0–10000
- `nb_clients` : slider 1–500
- Affichez `CAC = budget / nb_clients` et indiquez si c'est "faible", "moyen" ou "élevé".

In [ ]:
# Votre code ici


### Exercice 3
Créez un mini-formulaire de création de campagne avec un bouton "Créer". Au clic, affichez un dictionnaire récapitulatif. Utilisez `widgets.Text`, `Dropdown` et `Button`.

In [ ]:
# Votre code ici


### Exercice 4 — Validateur d'email 📧
Créez un mini-validateur d'adresse email :
- un `widgets.Text` pour saisir un email
- un `widgets.Button` **"Vérifier"**
- au clic, affichez (dans un `widgets.Output`) :
  - "✅ Email valide" si l'email contient à la fois `@` ET `.`
  - "❌ Email invalide" sinon

In [ ]:
# Votre code ici


---
## Résumé

Dans ce module, vous avez appris :
- À créer des widgets dans un notebook (sliders, dropdowns, boutons…)
- Le décorateur `@interact` pour rendre une fonction interactive en 1 ligne
- À organiser une interface avec `VBox` et `HBox`
- À réagir à un clic de bouton avec `.on_click()`

**Limite des notebooks :** vos interfaces ne fonctionnent que dans Jupyter, sur votre machine. Pour partager une vraie **application web** avec un client ou un manager, on passe à **Streamlit** dans le module suivant.

➡️ Et pour s'amuser un peu, le **module 9** propose deux mini-jeux interactifs (devine le nombre, pierre/feuille/ciseaux) qui réutilisent ces widgets.